# Tutorial: Saving Strategies and Data Standards

The following demonstrates how to use the `ep.saving_strategies` to save data in different formats utilizing data standards.

We will use the loaded variables from the cdf file to showcase how to save them with different formats. We will add additional variables so that we have more to save.

In [ ]:
from datetime import datetime, timezone

from astropy import units as u

import el_paso as ep

ep.setup_logging()

extraction_infos = [
    ep.ExtractionInfo(
        result_key="Epoch",
        name_or_column="Epoch_Ele",
        unit=ep.units.cdf_epoch,
    ),
    ep.ExtractionInfo(
        result_key="FEDU",
        name_or_column="FEDU",
        unit=(u.cm**2 * u.s * u.sr * u.keV) ** (-1),
    ),
    ep.ExtractionInfo(
        result_key="xGEO",
        name_or_column="Position_Ele",
        unit=u.km,
    ),
]

start_time = datetime(2017, 7, 14, tzinfo=timezone.utc)
end_time = datetime(2017, 7, 14, 23, 59, 59, tzinfo=timezone.utc)

file_name_stem = "rbspa_rel04_ect-hope-pa-l3_YYYYMMDD_.{6}.cdf"

ep.download(
    start_time,
    end_time,
    save_path=".",
    download_url="https://spdf.gsfc.nasa.gov/pub/data/rbsp/rbspa/l3/ect/hope/pitchangle/rel04/YYYY/",
    file_name_stem=file_name_stem,
    file_cadence="daily",
    method="request",
    skip_existing=True,
)

variables = ep.extract_variables_from_files(
    start_time, end_time, "daily", data_path=".", file_name_stem=file_name_stem, extraction_infos=extraction_infos
)
variables

# Saving Strategies

A saving strategy defines how the data is storred on disk. More specifically, it defines the folder structure, file names, cadence of the files (daily, monthly, etc), and which variables belong into which files. Different saving strategies allow us to save the same processed data in different formats according to our own needs. 

Examples:
- Save the data in either daily or monthly files
- Put all variables into a combined file or split them (e.g. variables associated with the orbit and variables associated with measurements).
- Use a pre-defined folder structure (e.g. mission/satellite/files)

## Single file strategy

First, we want to save the variables using the `SingleFileStrategy`. This is the simplest way to save variables, as everything is put into one file. This can be helpful for testing processing pipelines and debugging. Dependent on the file ending, different file formats will be saved. Possible formats are ".mat", ".nc", ".cdf" and ".h5".

The units of the variables are not changed when using the `SingleFileStrategy`.


In [ ]:
saving_strategy = ep.saving_strategies.SingleFileStrategy("rbsp_hope_example.mat")
ep.save(
    variables,
    saving_strategy=saving_strategy,
    start_time=start_time,
    end_time=end_time,
    time_var=variables["Epoch"],
    ignore_validation=True,
)

Let's inspect what got saved. The variables are turned into simple numpy arrays before saving. You can see that also a _metadata_ variable has been saved. We will look closer at metadata in a different tutorial.


In [ ]:
import scipy.io as sio

with open("rbsp_hope_example.mat", "rb") as f:
    loaded_data = sio.loadmat(f)
    print("Keys: ", loaded_data.keys())
    print("Metadata: ", loaded_data["metadata"])
    print("xGEO[0,:]: ", loaded_data["xGEO"][0, :] * u.Unit(loaded_data["metadata"]["xGEO"][0, 0]["unit"][0, 0][0]))

## Data Standards

Other saving strategies require a defined *Data standard*. A data standard defines how the data is organized **inside** the files. It defines the variable names, units, and dimensions. Before saving, EL-PASO performs consistency checks whether the data to be saved is complient with the given data standard (units are converted automatically, but saving will fail if the unit conversion fails).

Let's look next at some examples for saving strategies.

## MonthlyRBStrategy

Under this strategy, the data is stored in monthly files and all the variables are written into one combined monthly file. It is the strategy, you usually want to use when preparing data for radiation belt modelling (for high-cadence data, e.g. LEO, daily files might be the better choice).

The saving strategies need to know which variable in your script corresponds to which variable in the data standard. Therefore, we have to provide a dictionary (variables_to_save) which maps the internal names (`InternalName`) to our local variables. Here, we use the standard which is currently employed at GFZ.

In [ ]:
variables_to_save: dict[ep.typing.InternalName, ep.Variable] = {
    "Epoch": variables["Epoch"],
    "Position": variables["xGEO"],
}

monthly_strategy_gfz = ep.saving_strategies.MonthlyRBStrategy(
    "./RBSP/monthly_strategy_gfz",
    mission="RBSP",
    satellite="rbspa",
    instrument="hope",
    mag_field="T89",
    data_standard=ep.data_standards.GFZStandard(),
)

ep.save(variables_to_save, monthly_strategy_gfz, start_time, end_time, time_var=variables["Epoch"])

Let's change the data standard to the standard recommended by the COSPAR PRBEM and save the data one more time and inspect their content and differences.

In [ ]:
import netCDF4
from matplotlib import pyplot as plt

monthly_strategy_prbem = ep.saving_strategies.MonthlyRBStrategy(
    "./RBSP/monthly_strategy_prbem",
    mission="RBSP",
    satellite="rbspa",
    instrument="hope",
    mag_field="T89",
    data_standard=ep.data_standards.PRBEMStandard(),
)
ep.save(variables_to_save, monthly_strategy_prbem, start_time, end_time, time_var=variables["Epoch"])

with netCDF4.Dataset("./RBSP/monthly_strategy_gfz/RBSP/rbspa/rbspa_hope_20170701to20170731_T89.nc") as gfz_data, \
     netCDF4.Dataset("./RBSP/monthly_strategy_prbem/RBSP/rbspa/rbspa_hope_20170701to20170731_T89.nc") as prbem_data:

     gfz_variables = gfz_data.variables
     prbem_variables = prbem_data.variables

     print(f"GFZ variables: {gfz_variables.keys()}")
     print(f"PRBEM variables: {prbem_variables.keys()}")

     print(f"GFZ unit of orbit coordinates: {gfz_variables["xGEO"].getncattr("units")}")
     print(f"PRBEM unit of orbit coordinates: {prbem_variables["Position"].getncattr("units")}")




So we see that the same data got saved in two different ways, although the folder structure and file name, as defined by the saving strategy, has not changed. The orbital coordinate variable is called `xGEO` in the GFZ data standard, whereas it is called `Position` in the PRBEM standard. The variables are also saved in different units.

We will see a more elegant way to read in the processed data sets in the next tutorial.